In [0]:
%sql
show catalogs;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS demo_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo_catalog.demo_schema;

In [0]:
%sql
SHOW SCHEMAS IN demo_catalog;

In [0]:
data = [
    (101, "Ravi", "Hyderabad", 30, "ravi@gmail.com", "IT"),
    (102, "Sita", "Chennai", 25, "sita@gmail.com", "HR"),
    (103, "John", "Bangalore", 35, "john@gmail.com", "Finance"),
    (104, "Priya", "Mumbai", 28, "priya@gmail.com", "IT")
]

columns = [
    "customer_id",
    "customer_name",
    "city",
    "age",
    "email",
    "department"
]

df = spark.createDataFrame(data, columns)

df.show()

In [0]:
data = [
    (101, "Ravi", "Hyderabad", 30, "ravi@gmail.com", "IT"),
    (102, "Sita", "Chennai", 25, "sita@gmail.com", "HR"),
    (103, "John", "Bangalore", 35, "john@gmail.com", "Finance"),
    (104, "Priya", "Mumbai", 28, "priya@gmail.com", "IT")
]

columns = [
    "customer_id",
    "customer_name",
    "city",
    "age",
    "email",
    "department"
]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("demo_catalog.demo_schema.customers")

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers;

In [0]:
%sql
INSERT INTO demo_catalog.demo_schema.customers
(customer_id, customer_name, city, age, email, department)
VALUES
(105, 'Kiran', 'Delhi', 29, 'kiran@gmail.com', 'IT'),
(106, 'Meena', 'Kochi', 31, 'meena@gmail.com', 'HR'),
(107, 'Vijay', 'Mumbai', 27, 'vijay@gmail.com', 'Sales');

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers
ORDER BY customer_id;

In [0]:
%sql
CREATE OR REPLACE TABLE demo_catalog.demo_schema.customers AS
SELECT DISTINCT *
FROM demo_catalog.demo_schema.customers;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE HISTORY demo_catalog.demo_schema.customers;

In [0]:
%sql
UPDATE demo_catalog.demo_schema.customers
SET city = 'Hyderabad'
WHERE customer_id = 104;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers
WHERE customer_id = 104;

In [0]:
%sql
DESCRIBE HISTORY demo_catalog.demo_schema.customers;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers
VERSION AS OF 3
ORDER BY customer_id;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers
VERSION AS OF 4
ORDER BY customer_id;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers
VERSION AS OF 2
ORDER BY customer_id;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers VERSION AS OF 4

EXCEPT

SELECT *
FROM demo_catalog.demo_schema.customers VERSION AS OF 3;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers VERSION AS OF 3

EXCEPT

SELECT *
FROM demo_catalog.demo_schema.customers VERSION AS OF 2;

In [0]:
%sql
RESTORE TABLE demo_catalog.demo_schema.customers
TO VERSION AS OF 2;

In [0]:
%sql
DESCRIBE HISTORY demo_catalog.demo_schema.customers;

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("demo_catalog.demo_schema.customers")

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION databricks-data-location;

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION `databricks-data-location`;

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "demo_catalog.demo_schema.customers_external",
        path="abfss://databricks-data@mythriazurelearning.dfs.core.windows.net/DemoSamples/customers_external/"
    )

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE DETAIL demo_catalog.demo_schema.customers_external;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
ORDER BY customer_id;

In [0]:
%sql
INSERT INTO demo_catalog.demo_schema.customers_external
(customer_id, customer_name, city, age, email, department)
VALUES
(108, 'Anil', 'Pune', 32, 'anil@gmail.com', 'Finance');

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
ORDER BY customer_id;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
VERSION AS OF 0
ORDER BY customer_id;

In [0]:
%sql
DELETE FROM demo_catalog.demo_schema.customers_external
WHERE customer_id = 108;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE HISTORY demo_catalog.demo_schema.customers_external;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
VERSION AS OF 1
ORDER BY customer_id;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
ORDER BY customer_id;

In [0]:
new_data = [
    (104, "Priya", "Hyderabad", 29, "priya_new@gmail.com", "IT"),
    (109, "Rahul", "Delhi", 26, "rahul@gmail.com", "HR")
]

columns = [
    "customer_id",
    "customer_name",
    "city",
    "age",
    "email",
    "department"
]

new_df = spark.createDataFrame(new_data, columns)

new_df.show()

In [0]:
new_df.createOrReplaceTempView("new_customers")

In [0]:
%sql
MERGE INTO demo_catalog.demo_schema.customers_external AS target
USING new_customers AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
  UPDATE SET
    target.customer_name = source.customer_name,
    target.city = source.city,
    target.age = source.age,
    target.email = source.email,
    target.department = source.department

WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    customer_name,
    city,
    age,
    email,
    department
  )
  VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.age,
    source.email,
    source.department
  );

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE HISTORY demo_catalog.demo_schema.customers_external;

In [0]:
%sql
OPTIMIZE demo_catalog.demo_schema.customers_external;

In [0]:
%sql
DESCRIBE HISTORY demo_catalog.demo_schema.customers_external;

In [0]:
%sql
SELECT *
FROM demo_catalog.demo_schema.customers_external
ORDER BY customer_id;

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("department") \
    .saveAsTable(
        "demo_catalog.demo_schema.customers_partitioned"
    )

In [0]:
%sql
DESCRIBE DETAIL demo_catalog.demo_schema.customers_partitioned;